In [1]:
import os
import nibabel as nib
import skimage.io as io
import numpy as np
import tensorflow as tf
import LoadTrainingSet_PreprocessingFour_SmallSizeImageThree_CutTrain

import matplotlib.pyplot as plt  # plt 用于显示图片
import matplotlib.image as mpimg  # mpimg 用于读取图片

from sklearn.preprocessing import MinMaxScaler
from PIL import Image
from pylab import *

from functools import reduce

import math
import random
import time
import datetime

from skimage.measure import compare_ssim

branch_name = 'FullReluDepthwiseConvolution11'

input_path = '/project/6010822/sym/MyBrats/Brats2018/MICCAI_BraTS_2018_Data_Validation/'
# input_path = '/lustre03/project/6010822/sym/MyBrats/Brats2018/MICCAI_BraTS_2018_Data_Validation/'

print(time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))
training_start_time = datetime.datetime.now()

print('Start loading dataset: ')
training_set = LoadTrainingSet_PreprocessingFour_SmallSizeImageThree_CutTrain.BratsTrainingSet()  # load data from hard disk to RAM

np.savetxt(('training_' + branch_name + '_imageorder0.txt'), training_set.image_order_0, fmt='%d')
np.savetxt(('training_' + branch_name + '_imageorder1.txt'), training_set.image_order_1, fmt='%d')
np.savetxt(('training_' + branch_name + '_imageorder2.txt'), training_set.image_order_2, fmt='%d')

###################################################################
# training
#################################################################
print('Start training: ')
sess = tf.Session()

img_H = training_set.image_high  # 168
img_W = training_set.image_width  # 200
img_H_s = training_set.image_high_s
img_H_e = training_set.image_high_e
img_W_s = training_set.image_width_s
img_W_e = training_set.image_width_e

flair_in = tf.placeholder("float", shape=[None, img_H, img_W, 1])
t1_in = tf.placeholder("float", shape=[None, img_H, img_W, 1])
t1ce_in = tf.placeholder("float", shape=[None, img_H, img_W, 1])
t2_in = tf.placeholder("float", shape=[None, img_H, img_W, 1])
seg_in = tf.placeholder("float", shape=[None, img_H, img_W, 1])
# test flag for batch norm
tst = tf.placeholder(tf.bool)
iter = tf.placeholder(tf.int32)


def generalized_dice_loss(pred, true, p=1, q=1, eps=1E-6):
    """pred and true are tensors of shape (b, w_0, w_1, ..., c) where
             b   ... batch size
             w_k ... width of input in k-th dimension
             c   ... number of segments/classes
       Furthermore, boths tensors have exclusively values in [0, 1].
       more than already good ones. The remaining parameters are as follows:
             p   ... power of inverse weigthing (p=2 default, p=0 uniform)
             q   ... power of inverse loss weighting (q=1 default, q=0 none)
             eps ... regularization term if empty classes occur"""

    assert (p >= 0)
    assert (q >= 0)
    assert (eps >= 0)
    assert (pred.get_shape()[1:] == true.get_shape()[1:])

    m = "the values in your last layer must be strictly in [0, 1]"
    with tf.control_dependencies([]):

        shape_pred = pred.get_shape()
        shape_true = true.get_shape()
        prod_pred = reduce(lambda x, y: x * y, shape_pred[1:-1], tf.Dimension(1))
        prod_true = reduce(lambda x, y: x * y, shape_true[1:-1], tf.Dimension(1))

        # reshape to shape (b, W, c) where W is product of w_k
        pred = tf.reshape(pred, [-1, prod_pred, shape_pred[-1]])
        true = tf.reshape(true, [-1, prod_true, shape_true[-1]])

        # no class reweighting at all
        if p == 0:
            # unweighted intersection and union
            inter = tf.reduce_mean(pred * true, axis=[1, 2])
            union = tf.reduce_mean(pred + true, axis=[1, 2])
        else:
            # inverse L_p weighting for class cardinalities
            weights = tf.abs(tf.reduce_sum(true, axis=[1])) ** p + eps
            weights = tf.expand_dims(tf.reduce_sum(weights, axis=[-1]), -1) \
                      / weights

            # weighted intersection and union
            inter = tf.reduce_mean(weights * tf.reduce_mean(pred * true, axis=[1]),
                                   axis=[-1])
            union = tf.reduce_mean(weights * tf.reduce_mean(pred + true, axis=[1]),
                                   axis=[-1])

        # the traditional dice formula
        loss = 1.0 - 2.0 * (inter + eps) / (union + eps)

        # no reweighting of the batch
        if q == 0:
            return tf.reduce_mean(loss)

        # inverse L_q weighting for loss scores
        weights = tf.abs(loss) ** q + eps
        weights = tf.reduce_sum(weights) / weights

        return tf.reduce_mean(loss * weights) / tf.reduce_mean(weights)


def batchnorm(Ylogits, is_test, iteration, offset, convolutional=False):
    exp_moving_avg = tf.train.ExponentialMovingAverage(0.999,
                                                       iteration)  # adding the iteration prevents from averaging across non-existing iterations
    bnepsilon = 1e-5
    if convolutional:
        mean, variance = tf.nn.moments(Ylogits, [0, 1, 2])
    else:
        mean, variance = tf.nn.moments(Ylogits, [0])
    update_moving_averages = exp_moving_avg.apply([mean, variance])
    m = tf.cond(is_test, lambda: exp_moving_avg.average(mean), lambda: mean)
    v = tf.cond(is_test, lambda: exp_moving_avg.average(variance), lambda: variance)
    Ybn = tf.nn.batch_normalization(Ylogits, m, v, offset, None, bnepsilon)
    return Ybn, update_moving_averages


def no_batchnorm(Ylogits, is_test, iteration, offset, convolutional=False):
    return Ylogits, tf.no_op()


def instance_norm(x):
    mean, variance = tf.nn.moments(x, axes=[1,2], keep_dims=True)
    epsilon = 1e-5
    inv = tf.rsqrt(variance + epsilon)
    normalized = (x-mean)*inv
    return normalized


def weight_variable(shape):
    initial = tf.truncated_normal(shape, stddev=0.1)
    return tf.Variable(initial)


def bias_variable(shape):
    initial = tf.constant(0.1, shape=shape)
    return tf.Variable(initial)


def conv2d(x, W):
    return tf.nn.conv2d(x, W, strides=[1, 1, 1, 1], padding='SAME')


def depthwise_conv2d(x, W):
    return tf.nn.depthwise_conv2d(x, W, strides=[1, 1, 1, 1], padding='SAME')


def max_pool_2x2(x):
    return tf.nn.max_pool(x, ksize=[1, 2, 2, 1], strides=[1, 2, 2, 1], padding='SAME')


'''
def conv2d_transpose(x, w):
#    shape_x = x.get_shape().as_list()
    shape_w = w.get_shape().as_list()
    inputs_shape = tf.shape(x)
    outputs_shape = [inputs_shape[0], inputs_shape[1], inputs_shape[2], shape_w[2]]
#    shape_y = [x[0], shape_x[1] * 2, shape_x[2] * 2, shape_w[2]]
    r_Y = tf.nn.conv2d_transpose(x, w, output_shape=outputs_shape, strides=[1, 2, 2, 1], padding="SAME")
    return r_Y
'''


def full_relu(x):
    x1 = tf.nn.relu(x)
    x2 = tf.nn.relu(-x)
    return tf.concat([x1, x2], 3)


input_image_concat = tf.concat([flair_in, t1_in, t1ce_in, t2_in], 3)


### layer1_1
W_conv1_1 = weight_variable([3, 3, 4, 2])
b_conv1_1 = bias_variable([8])

h_conv1_1 = depthwise_conv2d(input_image_concat, W_conv1_1)

h_bn1_1 = instance_norm(h_conv1_1) + b_conv1_1

h_relu1_1 = full_relu(h_bn1_1)
h_pool1_1 = max_pool_2x2(h_relu1_1)

### layer1_2

W_conv1_2 = weight_variable([3, 3, 4, 8])
b_conv1_2 = bias_variable([8])

h_conv1_2 = conv2d(input_image_concat, W_conv1_2)
h_bn1_2, update_ema1_2 = batchnorm(h_conv1_2, tst, iter, b_conv1_2, convolutional=True)
h_relu1_2 = full_relu(h_bn1_2)
h_pool1_2 = max_pool_2x2(h_relu1_2)

h_relu1 = tf.concat([h_relu1_1, h_relu1_2], 3)
h_pool1 = tf.concat([h_pool1_1, h_pool1_2], 3)

# convolution + BN + relu + pool
### layer2_1
W_conv2_1 = weight_variable([3, 3, 16, 1])
b_conv2_1 = bias_variable([16])

h_conv2_1 = depthwise_conv2d(h_pool1_1, W_conv2_1)

h_bn2_1 = instance_norm(h_conv2_1) + b_conv2_1

h_relu2_1 = full_relu(h_bn2_1)
h_pool2_1 = max_pool_2x2(h_relu2_1)

# convolution + BN + relu + pool
### layer2_2
W_conv2_2 = weight_variable([3, 3, 16, 8])
b_conv2_2 = bias_variable([8])

h_conv2_2 = conv2d(h_pool1_2, W_conv2_2)
h_bn2_2, update_ema2_2 = batchnorm(h_conv2_2, tst, iter, b_conv2_2, convolutional=True)
h_relu2_2 = full_relu(h_bn2_2)
h_pool2_2 = max_pool_2x2(h_relu2_2)

h_relu2 = tf.concat([h_relu2_1, h_relu2_2], 3)
h_pool2 = tf.concat([h_pool2_1, h_pool2_2], 3)

# convolution + BN + relu + pool
W_conv3 = weight_variable([3, 3, 48, 16])
b_conv3 = bias_variable([16])

h_conv3 = conv2d(h_pool2, W_conv3)
h_bn3, update_ema3 = batchnorm(h_conv3, tst, iter, b_conv3, convolutional=True)
h_relu3 = full_relu(h_bn3)
h_pool3 = max_pool_2x2(h_relu3)

#############################
W_conv4 = weight_variable([3, 3, 32, 16])
b_conv4 = bias_variable([16])

h_conv4 = conv2d(h_pool3, W_conv4)
h_bn4, update_ema4 = batchnorm(h_conv4, tst, iter, b_conv4, convolutional=True)
h_relu4 = tf.nn.relu(h_bn4)
###################################


# upsample + conv + BN + relu
W_conv5 = weight_variable([3, 3, 16, 16])
b_conv5 = bias_variable([16])

h_upsample5 = tf.image.resize_images(h_relu4, [int(img_H / 4), int(img_W / 4)], 0)
W_conv5_1 = weight_variable([1, 1, 32, 16])
b_conv5_1 = bias_variable([16])
h_conv5_1 = conv2d(h_relu3, W_conv5_1) + b_conv5_1
h_upsample5 = h_upsample5 + h_conv5_1

h_conv5 = conv2d(h_upsample5, W_conv5)
h_bn5, update_ema5 = batchnorm(h_conv5, tst, iter, b_conv5, convolutional=True)
h_relu5 = tf.nn.relu(h_bn5)

# upsample + conv + BN + relu
W_conv6 = weight_variable([3, 3, 16, 16])
b_conv6 = bias_variable([16])

h_upsample6 = tf.image.resize_images(h_relu5, [int(img_H / 2), int(img_W / 2)], 0)
W_conv6_1 = weight_variable([1, 1, 48, 16])
b_conv6_1 = bias_variable([16])
h_conv6_1 = conv2d(h_relu2, W_conv6_1) + b_conv6_1
h_upsample6 = h_upsample6 + h_conv6_1

h_conv6 = conv2d(h_upsample6, W_conv6)
h_bn6, update_ema6 = batchnorm(h_conv6, tst, iter, b_conv6, convolutional=True)
h_relu6 = tf.nn.relu(h_bn6)

# upsample + conv + BN + relu
W_conv7 = weight_variable([3, 3, 16, 4])
b_conv7 = bias_variable([4])

h_upsample7 = tf.image.resize_images(h_relu6, [int(img_H), int(img_W)], 0)
W_conv7_1 = weight_variable([1, 1, 32, 16])
b_conv7_1 = bias_variable([16])
h_conv7_1 = conv2d(h_relu1, W_conv7_1) + b_conv7_1
h_upsample7 = h_upsample7 + h_conv7_1

h_conv7 = conv2d(h_upsample7, W_conv7) + b_conv7
# h_bn6, update_ema6 = batchnorm(h_conv6, tst, iter, b_conv6, convolutional=True)
# h_relu6 = tf.nn.relu(h_bn6)

#####################################################
# preprocess the ground true

# shape_seg_in = seg_in.get_shape().as_list()
# shape_seg_in[0] = -1
temp_zero = seg_in * 0.0
temp_one = temp_zero + 1.0

# 10 times error penalty for label 1, 2 and 4
seg_in_label_0 = tf.where(tf.equal(seg_in, temp_zero), temp_one, temp_zero)
seg_in_label_1 = tf.where(tf.equal(seg_in, temp_one), temp_one, temp_zero)
seg_in_label_2 = tf.where(tf.equal(seg_in, 2.0 * temp_one), temp_one, temp_zero)
seg_in_label_4 = tf.where(tf.equal(seg_in, 4.0 * temp_one), temp_one, temp_zero)
seg_in_label = tf.concat([seg_in_label_0, seg_in_label_1, seg_in_label_2, seg_in_label_4], 3)

##########################################

accuracy_list = []  # 保存准确率序列
cross_entropy_list = []
learning_rate_list = []
step_list = []
flip_b = -1  # flip training-set images: 0 no flip, 1 Up-Down flip, 2 Left-Right flip
n = 22101  # 训练次数
s = 50  # plot输出步长
batch_size = 100

#########################################
y_softmax = tf.nn.softmax(h_conv7, axis=3)
# cross_entropy = -tf.reduce_sum(seg_in_label * tf.log(y_softmax))
cross_entropy = tf.nn.softmax_cross_entropy_with_logits(logits=h_conv7, labels=seg_in_label)
cross_entropy = tf.reduce_mean(cross_entropy) * 100000
# dice_loss = generalized_dice_loss(y_softmax, seg_in_label, p=1, q=1, eps=1E-6)

# the learning rate is: # 0.00001 + 0.03 * (1/e)^(step/1000)), i.e. exponential decay from 0.03->0.0001
# lr = 0.000005 + tf.train.exponential_decay(0.0001, iter, 1000, 1 / math.e)

lr = tf.train.cosine_decay(learning_rate=0.01, global_step=iter, decay_steps=19900, alpha=0.0001)
train_step = tf.train.AdamOptimizer(lr).minimize(cross_entropy)
predict_image = tf.argmax(y_softmax, 3)  # if label equal 3, change to 4.
correct_prediction = tf.equal(predict_image, tf.argmax(seg_in_label, 3))
accuracy_pixel = tf.reduce_mean(tf.cast(correct_prediction, "float"))

update_ema = tf.group(update_ema1_2, update_ema2_2,
                      update_ema3, update_ema4, update_ema5, update_ema6)

#####################################################
######################################################

sess.run(tf.global_variables_initializer())

for i in range(n):
    if i % 500 == 0:
        flip_b = flip_b + 1
        if flip_b >= 3:
            flip_b = 0

    r_image_flair, r_image_t1, r_image_t1ce, r_image_t2, r_image_seg = training_set.next_batch(batch_size, flip_b)

    if i % s == 0:
        training_accuracy = sess.run(accuracy_pixel, feed_dict={flair_in: r_image_flair, t1_in: r_image_t1,
                                                                t1ce_in: r_image_t1ce, t2_in: r_image_t2,
                                                                seg_in: r_image_seg, tst: False, iter: i})
        training_cross_entropy = sess.run(cross_entropy, feed_dict={flair_in: r_image_flair, t1_in: r_image_t1,
                                                                    t1ce_in: r_image_t1ce, t2_in: r_image_t2,
                                                                    seg_in: r_image_seg, tst: False, iter: i})
        learning_rate = sess.run(lr, feed_dict={flair_in: r_image_flair, t1_in: r_image_t1,
                                                t1ce_in: r_image_t1ce, t2_in: r_image_t2,
                                                seg_in: r_image_seg, tst: False, iter: i})
        step_list.append(i)
        accuracy_list.append(training_accuracy)
        cross_entropy_list.append(training_cross_entropy)
        learning_rate_list.append(learning_rate)

        print("step %d, training accuracy %g, cross entropy %g, learning rate %g" % (
            i, training_accuracy, training_cross_entropy, learning_rate))
    sess.run(train_step, feed_dict={flair_in: r_image_flair, t1_in: r_image_t1,
                                    t1ce_in: r_image_t1ce, t2_in: r_image_t2,
                                    seg_in: r_image_seg, tst: False, iter: i})
    sess.run(update_ema, feed_dict={flair_in: r_image_flair, t1_in: r_image_t1,
                                    t1ce_in: r_image_t1ce, t2_in: r_image_t2,
                                    seg_in: r_image_seg, tst: False, iter: i})

print_content = [step_list, accuracy_list, cross_entropy_list, learning_rate_list]
print_content = list(map(list, zip(*print_content)))
np.savetxt(('training_' + branch_name + '_printcontent.txt'), print_content, fmt='%f')

print(time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))
training_end_time = datetime.datetime.now()
print('training time(s):', (training_end_time - training_start_time).seconds)

#########################################################################
# validation and submittion
###########################################################################
print('Start validation: ')

brats_validation_ID = ['Brats18_CBICA_AAM_1',
                       'Brats18_CBICA_ABT_1',
                       'Brats18_CBICA_ALA_1',
                       'Brats18_CBICA_ALT_1',
                       'Brats18_CBICA_ALV_1',
                       'Brats18_CBICA_ALZ_1',
                       'Brats18_CBICA_AMF_1',
                       'Brats18_CBICA_AMU_1',
                       'Brats18_CBICA_ANK_1',
                       'Brats18_CBICA_APM_1',
                       'Brats18_CBICA_AQE_1',
                       'Brats18_CBICA_ARR_1',
                       'Brats18_CBICA_ATW_1',
                       'Brats18_CBICA_AUC_1',
                       'Brats18_CBICA_AUE_1',
                       'Brats18_CBICA_AZA_1',
                       'Brats18_CBICA_BHF_1',
                       'Brats18_CBICA_BHN_1',
                       'Brats18_CBICA_BKY_1',
                       'Brats18_CBICA_BLI_1',
                       'Brats18_CBICA_BLK_1',
                       'Brats18_MDA_907_1',
                       'Brats18_MDA_922_1',
                       'Brats18_MDA_1012_1',
                       'Brats18_MDA_1015_1',
                       'Brats18_MDA_1081_1',
                       'Brats18_TCIA02_230_1',
                       'Brats18_TCIA02_400_1',
                       'Brats18_TCIA03_216_1',
                       'Brats18_TCIA03_288_1',
                       'Brats18_TCIA03_313_1',
                       'Brats18_TCIA03_604_1',
                       'Brats18_TCIA04_212_1',
                       'Brats18_TCIA04_253_1',
                       'Brats18_TCIA07_600_1',
                       'Brats18_TCIA07_601_1',
                       'Brats18_TCIA07_602_1',
                       'Brats18_TCIA09_248_1',
                       'Brats18_TCIA10_195_1',
                       'Brats18_TCIA10_311_1',
                       'Brats18_TCIA10_609_1',
                       'Brats18_TCIA11_612_1',
                       'Brats18_TCIA12_613_1',
                       'Brats18_TCIA13_610_1',
                       'Brats18_TCIA13_611_1',
                       'Brats18_TCIA13_617_1',
                       'Brats18_TCIA13_636_1',
                       'Brats18_TCIA13_638_1',
                       'Brats18_TCIA13_646_1',
                       'Brats18_TCIA13_652_1',
                       'Brats18_UAB_3446_1',
                       'Brats18_UAB_3448_1',
                       'Brats18_UAB_3449_1',
                       'Brats18_UAB_3454_1',
                       'Brats18_UAB_3455_1',
                       'Brats18_UAB_3456_1',
                       'Brats18_UAB_3490_1',
                       'Brats18_UAB_3498_1',
                       'Brats18_UAB_3499_1',
                       'Brats18_WashU_S036_1',
                       'Brats18_WashU_S037_1',
                       'Brats18_WashU_S041_1',
                       'Brats18_WashU_W033_1',
                       'Brats18_WashU_W038_1',
                       'Brats18_WashU_W047_1',
                       'Brats18_WashU_W053_1',
                       ]


def extract_patch(input_data, image_high, image_width, image_high_s, image_high_e, image_width_s, image_width_e):
    shape_data = np.shape(input_data)
    image_output = np.zeros((image_high, image_width, shape_data[2]), dtype=np.float)

    for j in range(shape_data[2]):
        image_output[:, :, j] = input_data[image_high_s:image_high_e, image_width_s:image_width_e, j]

    return image_output


def normalization(data):
    _range = np.max(data) - np.min(data)
    if _range == 0:
        return_data = data - np.min(data)
    else:
        return_data = (data - np.min(data)) / _range
    return return_data


def standardization(data):
    mu = np.mean(data)
    sigma = np.std(data)
    if sigma == 0:
        return_data = (data - mu)
    else:
        return_data = (data - mu) / sigma
    return return_data


def brats_validation(input_dir_path, output_dir_path, patient_ID):
    image_flair = np.zeros((155, img_H, img_W, 1), dtype=np.float)
    image_t1 = np.zeros((155, img_H, img_W, 1), dtype=np.float)
    image_t1ce = np.zeros((155, img_H, img_W, 1), dtype=np.float)
    image_t2 = np.zeros((155, img_H, img_W, 1), dtype=np.float)

    image_predict = np.zeros((240, 240, 155), dtype=np.float)

    path_flair = os.path.join(input_dir_path + patient_ID + '/' + patient_ID + '_flair.nii.gz')
    path_t1 = os.path.join(input_dir_path + patient_ID + '/' + patient_ID + '_t1.nii.gz')
    path_t1ce = os.path.join(input_dir_path + patient_ID + '/' + patient_ID + '_t1ce.nii.gz')
    path_t2 = os.path.join(input_dir_path + patient_ID + '/' + patient_ID + '_t2.nii.gz')

    flair_temp = nib.load(path_flair)
    flair_temp_arr = flair_temp.get_fdata()
    flair_temp_arr_sq = np.squeeze(flair_temp_arr)
    flair_temp_arr_sq = extract_patch(flair_temp_arr_sq, img_H, img_W, img_H_s, img_H_e, img_W_s, img_W_e)

    t1_temp = nib.load(path_t1)
    t1_temp_arr = t1_temp.get_fdata()
    t1_temp_arr_sq = np.squeeze(t1_temp_arr)
    t1_temp_arr_sq = extract_patch(t1_temp_arr_sq, img_H, img_W, img_H_s, img_H_e, img_W_s, img_W_e)

    t1ce_temp = nib.load(path_t1ce)
    t1ce_temp_arr = t1ce_temp.get_fdata()
    t1ce_temp_arr_sq = np.squeeze(t1ce_temp_arr)
    t1ce_temp_arr_sq = extract_patch(t1ce_temp_arr_sq, img_H, img_W, img_H_s, img_H_e, img_W_s, img_W_e)

    t2_temp = nib.load(path_t2)
    t2_temp_arr = t2_temp.get_fdata()
    t2_temp_arr_sq = np.squeeze(t2_temp_arr)
    t2_temp_arr_sq = extract_patch(t2_temp_arr_sq, img_H, img_W, img_H_s, img_H_e, img_W_s, img_W_e)

    affine_array = flair_temp.affine

    ######  Normalization and Standardization  ########

    flair_temp_arr_sq = normalization(flair_temp_arr_sq)
    flair_temp_arr_sq = standardization(flair_temp_arr_sq)

    t1_temp_arr_sq = normalization(t1_temp_arr_sq)
    t1_temp_arr_sq = standardization(t1_temp_arr_sq)

    t1ce_temp_arr_sq = normalization(t1ce_temp_arr_sq)
    t1ce_temp_arr_sq = standardization(t1ce_temp_arr_sq)

    t2_temp_arr_sq = normalization(t2_temp_arr_sq)
    t2_temp_arr_sq = standardization(t2_temp_arr_sq)

    ###################################################

    for i in range(155):
        image_flair[i, :, :, 0] = flair_temp_arr_sq[:, :, i]
        image_t1[i, :, :, 0] = t1_temp_arr_sq[:, :, i]
        image_t1ce[i, :, :, 0] = t1ce_temp_arr_sq[:, :, i]
        image_t2[i, :, :, 0] = t2_temp_arr_sq[:, :, i]

    batch_flair = np.zeros((1, img_H, img_W, 1), dtype=np.float)
    batch_t1 = np.zeros((1, img_H, img_W, 1), dtype=np.float)
    batch_t1ce = np.zeros((1, img_H, img_W, 1), dtype=np.float)
    batch_t2 = np.zeros((1, img_H, img_W, 1), dtype=np.float)

    for i in range(155):
        batch_flair[0, :, :, 0] = image_flair[i, :, :, 0]
        batch_t1[0, :, :, 0] = image_t1[i, :, :, 0]
        batch_t1ce[0, :, :, 0] = image_t1ce[i, :, :, 0]
        batch_t2[0, :, :, 0] = image_t2[i, :, :, 0]

        t_predict_image = sess.run(predict_image, feed_dict={flair_in: batch_flair, t1_in: batch_t1,
                                                             t1ce_in: batch_t1ce, t2_in: batch_t2,
                                                             tst: True})
        t_predict_image_p = np.squeeze(t_predict_image)
        t_predict_image_p[t_predict_image_p == 3] = 4
        image_predict[img_H_s:img_H_e, img_W_s:img_W_e, i] = t_predict_image_p[:, :]

    image_predict = image_predict.astype(int16)
    new_image_predict = nib.Nifti1Image(image_predict, affine_array)
    nib.save(new_image_predict, os.path.join(output_dir_path + patient_ID + '.nii.gz'))


num_validation_patient = len(brats_validation_ID)
# dir_path = 'C:/Users/whsym/Desktop/pythonCode/MyBrats/Brats2018/MICCAI_BraTS_2018_Data_Validation/'


output_path = './Validation_' + branch_name + '_submit/'
if not os.path.exists(output_path):
    os.makedirs(output_path)

for j in range(num_validation_patient):
    brats_validation(input_path, output_path, brats_validation_ID[j])

    if j == round(num_validation_patient / 4):
        print('rate of progress 25%')
    if j == round(num_validation_patient / 2):
        print('rate of progress 50%')
    if j == round(num_validation_patient / 4 * 3):
        print('rate of progress 75%')
    if j == num_validation_patient - 1:
        print('rate of progress 100%')

print(time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))
validation_end_time = datetime.datetime.now()
print('validation time(s):', (validation_end_time - training_end_time).seconds)


######################################################
# validation (cut image based on contour , ssim and background)
######################################################


def FlipImage_UpDown(image):
    shape_image = np.shape(image)
    temp_image = np.zeros((shape_image[0], shape_image[1]), dtype=np.float)
    for i in range(shape_image[0]):
        temp_image[i, :] = image[shape_image[0] - 1 - i, :]
    return temp_image


def count_pixel(arr, target):
    mask = (arr == target)
    arr_new = arr[mask]
    return arr_new.size


def count_contour_ssim(image):
    if image.max() > 0:
        v_start = 0
        v_end = 0
        h_start = 0
        h_end = 0
        shape_image = np.shape(image)
        for k in range(shape_image[0]):
            if image[k, :].max() > 0:
                v_start = k
                break
        for k in range(shape_image[0]):
            if image[int(shape_image[0] - k - 1), :].max() > 0:
                v_end = int(shape_image[0] - k - 1)
                break
        for k in range(shape_image[1]):
            if image[:, k].max() > 0:
                h_start = k
                break
        for k in range(shape_image[1]):
            if image[:, int(shape_image[1] - k - 1)].max() > 0:
                h_end = int(shape_image[1] - k - 1)
                break
        v_start = min([shape_image[0] - 1 - v_end + 1, v_start - 0 + 1]) - 1
        v_end = shape_image[0] - 1 - v_start

        image_s_shape = [int(v_end - v_start + 1), int(h_end - h_start + 1)]

        if image_s_shape[0] > 15 and image_s_shape[1] > 15:
            #            image_s = np.zeros((image_s_shape[0], image_s_shape[1]), dtype=np.float)
            image_s_up1 = np.zeros((int(image_s_shape[0] / 2), image_s_shape[1]), dtype=np.float)
            image_s_down1 = np.zeros((int(image_s_shape[0] / 2), image_s_shape[1]), dtype=np.float)
            image_s_up2 = np.zeros((int(image_s_shape[0] / 2), image_s_shape[1]), dtype=np.float)
            image_s_down2 = np.zeros((int(image_s_shape[0] / 2), image_s_shape[1]), dtype=np.float)

            #            image_s[:, :] = image[v_start:v_end+1, h_start:h_end+1]
            image_s_up1[:, :] = image[v_start:int(shape_image[0] / 2), h_start:h_end + 1]
            image_s_up1[:, :] = FlipImage_UpDown(image_s_up1[:, :])
            image_s_down1[:, :] = image[int(shape_image[0] / 2):v_end + 1, h_start:h_end + 1]

            image_s_up2[:, :] = image_s_up1[:, :]
            image_s_down2[:, :] = image_s_down1[:, :]

            image_s_up1[image_s_up1 != 0] = 255
            image_s_down1[image_s_down1 != 0] = 255

            #            print(np.shape(image_s_up1))
            #            print(np.shape(image_s_down1))
            contour_ssim = compare_ssim(image_s_up1, image_s_down1)

            signal_ssim = compare_ssim(image_s_up2, image_s_down2)

            return contour_ssim, signal_ssim
        else:
            return 1, 1
    else:
        return 1, 1


def brats_validation_cut(input_dir_path, output_dir_path, patient_ID):
    image_flair = np.zeros((155, img_H, img_W, 1), dtype=np.float)
    image_t1 = np.zeros((155, img_H, img_W, 1), dtype=np.float)
    image_t1ce = np.zeros((155, img_H, img_W, 1), dtype=np.float)
    image_t2 = np.zeros((155, img_H, img_W, 1), dtype=np.float)

    image_predict = np.zeros((240, 240, 155), dtype=np.float)

    path_flair = os.path.join(input_dir_path + patient_ID + '/' + patient_ID + '_flair.nii.gz')
    path_t1 = os.path.join(input_dir_path + patient_ID + '/' + patient_ID + '_t1.nii.gz')
    path_t1ce = os.path.join(input_dir_path + patient_ID + '/' + patient_ID + '_t1ce.nii.gz')
    path_t2 = os.path.join(input_dir_path + patient_ID + '/' + patient_ID + '_t2.nii.gz')

    flair_temp = nib.load(path_flair)
    flair_temp_arr = flair_temp.get_fdata()
    flair_temp_arr_sq = np.squeeze(flair_temp_arr)
    flair_temp_arr_sq = extract_patch(flair_temp_arr_sq, img_H, img_W, img_H_s, img_H_e, img_W_s, img_W_e)

    t1_temp = nib.load(path_t1)
    t1_temp_arr = t1_temp.get_fdata()
    t1_temp_arr_sq = np.squeeze(t1_temp_arr)
    t1_temp_arr_sq = extract_patch(t1_temp_arr_sq, img_H, img_W, img_H_s, img_H_e, img_W_s, img_W_e)

    t1ce_temp = nib.load(path_t1ce)
    t1ce_temp_arr = t1ce_temp.get_fdata()
    t1ce_temp_arr_sq = np.squeeze(t1ce_temp_arr)
    t1ce_temp_arr_sq = extract_patch(t1ce_temp_arr_sq, img_H, img_W, img_H_s, img_H_e, img_W_s, img_W_e)

    t2_temp = nib.load(path_t2)
    t2_temp_arr = t2_temp.get_fdata()
    t2_temp_arr_sq = np.squeeze(t2_temp_arr)
    t2_temp_arr_sq = extract_patch(t2_temp_arr_sq, img_H, img_W, img_H_s, img_H_e, img_W_s, img_W_e)

    affine_array = flair_temp.affine

    ###################### calculate cutting condition

    background_flair = np.zeros(155, dtype=np.float)
    for k in range(155):
        background_flair[k] = count_pixel(flair_temp_arr_sq[:, :, k], 0)
        background_flair[k] = background_flair[k] / img_H / img_W

    contour_ssim_flair = np.zeros(155, dtype=np.float)
    signal_ssim_flair = np.zeros(155, dtype=np.float)

    for i in range(155):
        contour_ssim_flair[i], signal_ssim_flair[i] = count_contour_ssim(flair_temp_arr_sq[:, :, i])

    ######################

    ######  Normalization and Standardization  ########

    flair_temp_arr_sq = normalization(flair_temp_arr_sq)
    flair_temp_arr_sq = standardization(flair_temp_arr_sq)

    t1_temp_arr_sq = normalization(t1_temp_arr_sq)
    t1_temp_arr_sq = standardization(t1_temp_arr_sq)

    t1ce_temp_arr_sq = normalization(t1ce_temp_arr_sq)
    t1ce_temp_arr_sq = standardization(t1ce_temp_arr_sq)

    t2_temp_arr_sq = normalization(t2_temp_arr_sq)
    t2_temp_arr_sq = standardization(t2_temp_arr_sq)

    ###################################################

    for i in range(155):
        image_flair[i, :, :, 0] = flair_temp_arr_sq[:, :, i]
        image_t1[i, :, :, 0] = t1_temp_arr_sq[:, :, i]
        image_t1ce[i, :, :, 0] = t1ce_temp_arr_sq[:, :, i]
        image_t2[i, :, :, 0] = t2_temp_arr_sq[:, :, i]

    batch_flair = np.zeros((1, img_H, img_W, 1), dtype=np.float)
    batch_t1 = np.zeros((1, img_H, img_W, 1), dtype=np.float)
    batch_t1ce = np.zeros((1, img_H, img_W, 1), dtype=np.float)
    batch_t2 = np.zeros((1, img_H, img_W, 1), dtype=np.float)

    number_slide = 0

    for i in range(155):
        if i <= 77 and background_flair[i] <= 0.90 and contour_ssim_flair[i] >= 0.35 and signal_ssim_flair[
            i] <= 0.58:
            batch_flair[0, :, :, 0] = image_flair[i, :, :, 0]
            batch_t1[0, :, :, 0] = image_t1[i, :, :, 0]
            batch_t1ce[0, :, :, 0] = image_t1ce[i, :, :, 0]
            batch_t2[0, :, :, 0] = image_t2[i, :, :, 0]

            t_predict_image = sess.run(predict_image, feed_dict={flair_in: batch_flair, t1_in: batch_t1,
                                                                 t1ce_in: batch_t1ce, t2_in: batch_t2,
                                                                 tst: True})
            t_predict_image_p = np.squeeze(t_predict_image)
            t_predict_image_p[t_predict_image_p == 3] = 4
            image_predict[img_H_s:img_H_e, img_W_s:img_W_e, i] = t_predict_image_p[:, :]

            number_slide = number_slide + 1

        if i > 77 and background_flair[i] < 1 and signal_ssim_flair[i] <= 0.58:
            batch_flair[0, :, :, 0] = image_flair[i, :, :, 0]
            batch_t1[0, :, :, 0] = image_t1[i, :, :, 0]
            batch_t1ce[0, :, :, 0] = image_t1ce[i, :, :, 0]
            batch_t2[0, :, :, 0] = image_t2[i, :, :, 0]

            t_predict_image = sess.run(predict_image, feed_dict={flair_in: batch_flair, t1_in: batch_t1,
                                                                 t1ce_in: batch_t1ce, t2_in: batch_t2,
                                                                 tst: True})
            t_predict_image_p = np.squeeze(t_predict_image)
            t_predict_image_p[t_predict_image_p == 3] = 4
            image_predict[img_H_s:img_H_e, img_W_s:img_W_e, i] = t_predict_image_p[:, :]

            number_slide = number_slide + 1

    image_predict = image_predict.astype(int16)
    new_image_predict = nib.Nifti1Image(image_predict, affine_array)
    nib.save(new_image_predict, os.path.join(output_dir_path + patient_ID + '.nii.gz'))

    return number_slide


# num_validation_patient = len(brats_validation_ID)
# dir_path = 'C:/Users/whsym/Desktop/pythonCode/MyBrats/Brats2018/MICCAI_BraTS_2018_Data_Validation/'


output_path_cut = './Validation_' + branch_name + '_submit_Cut/'
if not os.path.exists(output_path_cut):
    os.makedirs(output_path_cut)

slide_number = np.zeros(num_validation_patient, dtype=np.int)

print('start validation (cut)')
print('Patient_ID', 'number_slides')

for j in range(num_validation_patient):
    slide_number[j] = brats_validation_cut(input_path, output_path_cut, brats_validation_ID[j])
    print(brats_validation_ID[j], slide_number[j])

print('Average number of slides: ', np.mean(slide_number))

print(time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))
validation_cut_end_time = datetime.datetime.now()
print('validation_time_cut(s):', (validation_cut_end_time - validation_end_time).seconds)


######################################################
# validation (cut image based on contour , ssim and background; Post-processing)
######################################################
def post_processing(input_data, num_WT, num_ET, num_TC):
    shape_data = np.shape(input_data)

    pred_max = np.zeros(shape_data[2], dtype=np.float)
    output_data = np.zeros((shape_data[0], shape_data[1], shape_data[2]), dtype=np.float)

    for i in range(shape_data[2]):
        pred_max[i] = input_data[:, :, i].max()

    for i in range(shape_data[2]):
        num_index_WT = 0
        num_index_ET = 0
        num_index_TC = 0
        if i >= num_WT - 1 and i <= shape_data[2] - num_WT and pred_max[i] > 0:
            for j in range(num_WT):
                if pred_max[i - j] > 0:
                    num_index_WT = num_index_WT + 1
                else:
                    break
            for j in range(num_WT):
                if pred_max[i + j] > 0:
                    num_index_WT = num_index_WT + 1
                else:
                    break

            if num_index_WT - 1 >= num_WT:  # 上面两个循环把当前点算了两次
                output_data[:, :, i] = input_data[:, :, i]
            else:
                output_data[:, :, i] = 0

        if i >= num_ET - 1 and i <= shape_data[2] - num_ET and pred_max[i] == 4:
            for j in range(num_ET):
                if pred_max[i - j] == 4:
                    num_index_ET = num_index_ET + 1
                else:
                    break
            for j in range(num_ET):
                if pred_max[i + j] == 4:
                    num_index_ET = num_index_ET + 1
                else:
                    break
            if num_index_ET - 1 < num_ET:
                output_data[:, :, i][output_data[:, :, i] == 4] = 1

        if i >= num_TC - 1 and i <= shape_data[2] - num_TC and pred_max[i] == 1:
            for j in range(num_TC):
                if pred_max[i - j] == 1:
                    num_index_TC = num_index_TC + 1
                else:
                    break
            for j in range(num_TC):
                if pred_max[i + j] == 1:
                    num_index_TC = num_index_TC + 1
                else:
                    break
            if num_index_TC - 1 < num_TC:
                output_data[:, :, i][output_data[:, :, i] == 1] = 2

    return output_data


def brats_validation_cut_post(input_dir_path, output_dir_path, patient_ID):
    image_flair = np.zeros((155, img_H, img_W, 1), dtype=np.float)
    image_t1 = np.zeros((155, img_H, img_W, 1), dtype=np.float)
    image_t1ce = np.zeros((155, img_H, img_W, 1), dtype=np.float)
    image_t2 = np.zeros((155, img_H, img_W, 1), dtype=np.float)

    image_predict = np.zeros((240, 240, 155), dtype=np.float)

    path_flair = os.path.join(input_dir_path + patient_ID + '/' + patient_ID + '_flair.nii.gz')
    path_t1 = os.path.join(input_dir_path + patient_ID + '/' + patient_ID + '_t1.nii.gz')
    path_t1ce = os.path.join(input_dir_path + patient_ID + '/' + patient_ID + '_t1ce.nii.gz')
    path_t2 = os.path.join(input_dir_path + patient_ID + '/' + patient_ID + '_t2.nii.gz')

    flair_temp = nib.load(path_flair)
    flair_temp_arr = flair_temp.get_fdata()
    flair_temp_arr_sq = np.squeeze(flair_temp_arr)
    flair_temp_arr_sq = extract_patch(flair_temp_arr_sq, img_H, img_W, img_H_s, img_H_e, img_W_s, img_W_e)

    t1_temp = nib.load(path_t1)
    t1_temp_arr = t1_temp.get_fdata()
    t1_temp_arr_sq = np.squeeze(t1_temp_arr)
    t1_temp_arr_sq = extract_patch(t1_temp_arr_sq, img_H, img_W, img_H_s, img_H_e, img_W_s, img_W_e)

    t1ce_temp = nib.load(path_t1ce)
    t1ce_temp_arr = t1ce_temp.get_fdata()
    t1ce_temp_arr_sq = np.squeeze(t1ce_temp_arr)
    t1ce_temp_arr_sq = extract_patch(t1ce_temp_arr_sq, img_H, img_W, img_H_s, img_H_e, img_W_s, img_W_e)

    t2_temp = nib.load(path_t2)
    t2_temp_arr = t2_temp.get_fdata()
    t2_temp_arr_sq = np.squeeze(t2_temp_arr)
    t2_temp_arr_sq = extract_patch(t2_temp_arr_sq, img_H, img_W, img_H_s, img_H_e, img_W_s, img_W_e)

    affine_array = flair_temp.affine

    ###################### calculate cutting condition

    background_flair = np.zeros(155, dtype=np.float)
    for k in range(155):
        background_flair[k] = count_pixel(flair_temp_arr_sq[:, :, k], 0)
        background_flair[k] = background_flair[k] / img_H / img_W

    contour_ssim_flair = np.zeros(155, dtype=np.float)
    signal_ssim_flair = np.zeros(155, dtype=np.float)

    for i in range(155):
        contour_ssim_flair[i], signal_ssim_flair[i] = count_contour_ssim(flair_temp_arr_sq[:, :, i])

    ######################

    ######  Normalization and Standardization  ########

    flair_temp_arr_sq = normalization(flair_temp_arr_sq)
    flair_temp_arr_sq = standardization(flair_temp_arr_sq)

    t1_temp_arr_sq = normalization(t1_temp_arr_sq)
    t1_temp_arr_sq = standardization(t1_temp_arr_sq)

    t1ce_temp_arr_sq = normalization(t1ce_temp_arr_sq)
    t1ce_temp_arr_sq = standardization(t1ce_temp_arr_sq)

    t2_temp_arr_sq = normalization(t2_temp_arr_sq)
    t2_temp_arr_sq = standardization(t2_temp_arr_sq)

    ###################################################

    for i in range(155):
        image_flair[i, :, :, 0] = flair_temp_arr_sq[:, :, i]
        image_t1[i, :, :, 0] = t1_temp_arr_sq[:, :, i]
        image_t1ce[i, :, :, 0] = t1ce_temp_arr_sq[:, :, i]
        image_t2[i, :, :, 0] = t2_temp_arr_sq[:, :, i]

    batch_flair = np.zeros((1, img_H, img_W, 1), dtype=np.float)
    batch_t1 = np.zeros((1, img_H, img_W, 1), dtype=np.float)
    batch_t1ce = np.zeros((1, img_H, img_W, 1), dtype=np.float)
    batch_t2 = np.zeros((1, img_H, img_W, 1), dtype=np.float)

    number_slide = 0

    for i in range(155):
        if i <= 77 and background_flair[i] <= 0.90 and contour_ssim_flair[i] >= 0.35 and signal_ssim_flair[
            i] <= 0.58:
            batch_flair[0, :, :, 0] = image_flair[i, :, :, 0]
            batch_t1[0, :, :, 0] = image_t1[i, :, :, 0]
            batch_t1ce[0, :, :, 0] = image_t1ce[i, :, :, 0]
            batch_t2[0, :, :, 0] = image_t2[i, :, :, 0]

            t_predict_image = sess.run(predict_image, feed_dict={flair_in: batch_flair, t1_in: batch_t1,
                                                                 t1ce_in: batch_t1ce, t2_in: batch_t2,
                                                                 tst: True})
            t_predict_image_p = np.squeeze(t_predict_image)
            t_predict_image_p[t_predict_image_p == 3] = 4
            image_predict[img_H_s:img_H_e, img_W_s:img_W_e, i] = t_predict_image_p[:, :]

            number_slide = number_slide + 1

        if i > 77 and background_flair[i] < 1 and signal_ssim_flair[i] <= 0.58:
            batch_flair[0, :, :, 0] = image_flair[i, :, :, 0]
            batch_t1[0, :, :, 0] = image_t1[i, :, :, 0]
            batch_t1ce[0, :, :, 0] = image_t1ce[i, :, :, 0]
            batch_t2[0, :, :, 0] = image_t2[i, :, :, 0]

            t_predict_image = sess.run(predict_image, feed_dict={flair_in: batch_flair, t1_in: batch_t1,
                                                                 t1ce_in: batch_t1ce, t2_in: batch_t2,
                                                                 tst: True})
            t_predict_image_p = np.squeeze(t_predict_image)
            t_predict_image_p[t_predict_image_p == 3] = 4
            image_predict[img_H_s:img_H_e, img_W_s:img_W_e, i] = t_predict_image_p[:, :]

            number_slide = number_slide + 1

    image_post = post_processing(image_predict, 7, 6, 1)

    image_post = image_post.astype(int16)
    new_image_post = nib.Nifti1Image(image_post, affine_array)
    nib.save(new_image_post, os.path.join(output_dir_path + patient_ID + '.nii.gz'))


# num_validation_patient = len(brats_validation_ID)
# dir_path = 'C:/Users/whsym/Desktop/pythonCode/MyBrats/Brats2018/MICCAI_BraTS_2018_Data_Validation/'


output_path_cut_post = './Validation_' + branch_name + '_submit_CutPost/'
if not os.path.exists(output_path_cut_post):
    os.makedirs(output_path_cut_post)

print('start validation (cut and post)')

for j in range(num_validation_patient):
    brats_validation_cut_post(input_path, output_path_cut_post, brats_validation_ID[j])

    if j == round(num_validation_patient / 4):
        print('rate of progress 25%')
    if j == round(num_validation_patient / 2):
        print('rate of progress 50%')
    if j == round(num_validation_patient / 4 * 3):
        print('rate of progress 75%')
    if j == num_validation_patient - 1:
        print('rate of progress 100%')

print(time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))
validation_cutandpost_end_time = datetime.datetime.now()
print('validation_time_cut_post(s):', (validation_cutandpost_end_time - validation_cut_end_time).seconds)


###############################################################
# test training-set
##############################################################

def calculate_dice(predict_image, seg_image, patient_ID):
    predict_image = predict_image.astype(np.float)
    seg_image = seg_image.astype(np.float)

    shape_image = np.shape(seg_image)

    predict_image = predict_image.reshape((shape_image[0], shape_image[1] * shape_image[2]))
    seg_image = seg_image.reshape((shape_image[0], shape_image[1] * shape_image[2]))

    #################
    predict_ET = np.zeros((shape_image[0], shape_image[1] * shape_image[2]), dtype=np.float)
    seg_ET = np.zeros((shape_image[0], shape_image[1] * shape_image[2]), dtype=np.float)

    predict_WT = np.zeros((shape_image[0], shape_image[1] * shape_image[2]), dtype=np.float)
    seg_WT = np.zeros((shape_image[0], shape_image[1] * shape_image[2]), dtype=np.float)

    predict_TC = np.zeros((shape_image[0], shape_image[1] * shape_image[2]), dtype=np.float)
    seg_TC = np.zeros((shape_image[0], shape_image[1] * shape_image[2]), dtype=np.float)

    ###############
    predict_ET[:, :] = predict_image[:, :]
    predict_ET[predict_ET == 1] = 0
    predict_ET[predict_ET == 2] = 0
    predict_ET[predict_ET == 4] = 1
    seg_ET[:, :] = seg_image[:, :]
    seg_ET[seg_ET == 1] = 0
    seg_ET[seg_ET == 2] = 0
    seg_ET[seg_ET == 4] = 1

    predict_WT[:, :] = predict_image[:, :]
    predict_WT[predict_WT == 1] = 1
    predict_WT[predict_WT == 2] = 1
    predict_WT[predict_WT == 4] = 1
    seg_WT[:, :] = seg_image[:, :]
    seg_WT[seg_WT == 1] = 1
    seg_WT[seg_WT == 2] = 1
    seg_WT[seg_WT == 4] = 1

    predict_TC[:, :] = predict_image[:, :]
    predict_TC[predict_TC == 1] = 1
    predict_TC[predict_TC == 2] = 0
    predict_TC[predict_TC == 4] = 1
    seg_TC[:, :] = seg_image[:, :]
    seg_TC[seg_TC == 1] = 1
    seg_TC[seg_TC == 2] = 0
    seg_TC[seg_TC == 4] = 1

    ep = 0.0000001
    if np.sum(seg_ET) == 0:
        dice_ET = 1
    else:
        dice_ET = 2 * np.sum(seg_ET * predict_ET) / (np.sum(seg_ET) + np.sum(predict_ET) + ep)
    dice_WT = 2 * np.sum(seg_WT * predict_WT) / (np.sum(seg_WT) + np.sum(predict_WT) + ep)
    dice_TC = 2 * np.sum(seg_TC * predict_TC) / (np.sum(seg_TC) + np.sum(predict_TC) + ep)

    print(patient_ID, dice_ET, dice_WT, dice_TC)
    return dice_ET, dice_WT, dice_TC


def brats_test_trainingset(input_dir_path, output_dir_path, patient_path):
    image_flair = np.zeros((155, img_H, img_W, 1), dtype=np.float)
    image_t1 = np.zeros((155, img_H, img_W, 1), dtype=np.float)
    image_t1ce = np.zeros((155, img_H, img_W, 1), dtype=np.float)
    image_t2 = np.zeros((155, img_H, img_W, 1), dtype=np.float)

    image_predict = np.zeros((240, 240, 155), dtype=np.float)
    image_seg = np.zeros((240, 240, 155), dtype=np.float)

    zero_pixel_percent = np.zeros(155, dtype=np.float)

    path_flair = os.path.join(input_dir_path + patient_path + '/' + patient_path[4:len(patient_path)] + '_flair.nii.gz')
    path_t1 = os.path.join(input_dir_path + patient_path + '/' + patient_path[4:len(patient_path)] + '_t1.nii.gz')
    path_t1ce = os.path.join(input_dir_path + patient_path + '/' + patient_path[4:len(patient_path)] + '_t1ce.nii.gz')
    path_t2 = os.path.join(input_dir_path + patient_path + '/' + patient_path[4:len(patient_path)] + '_t2.nii.gz')
    path_seg = os.path.join(input_dir_path + patient_path + '/' + patient_path[4:len(patient_path)] + '_seg.nii.gz')

    flair_temp = nib.load(path_flair)
    flair_temp_arr = flair_temp.get_fdata()
    flair_temp_arr_sq = np.squeeze(flair_temp_arr)
    flair_temp_arr_sq = extract_patch(flair_temp_arr_sq, img_H, img_W, img_H_s, img_H_e, img_W_s, img_W_e)

    t1_temp = nib.load(path_t1)
    t1_temp_arr = t1_temp.get_fdata()
    t1_temp_arr_sq = np.squeeze(t1_temp_arr)
    t1_temp_arr_sq = extract_patch(t1_temp_arr_sq, img_H, img_W, img_H_s, img_H_e, img_W_s, img_W_e)

    t1ce_temp = nib.load(path_t1ce)
    t1ce_temp_arr = t1ce_temp.get_fdata()
    t1ce_temp_arr_sq = np.squeeze(t1ce_temp_arr)
    t1ce_temp_arr_sq = extract_patch(t1ce_temp_arr_sq, img_H, img_W, img_H_s, img_H_e, img_W_s, img_W_e)

    t2_temp = nib.load(path_t2)
    t2_temp_arr = t2_temp.get_fdata()
    t2_temp_arr_sq = np.squeeze(t2_temp_arr)
    t2_temp_arr_sq = extract_patch(t2_temp_arr_sq, img_H, img_W, img_H_s, img_H_e, img_W_s, img_W_e)

    seg_temp = nib.load(path_seg)
    seg_temp_arr = seg_temp.get_fdata()
    seg_temp_arr_sq = np.squeeze(seg_temp_arr)

    affine_array = flair_temp.affine

    ######  Normalization and Standardization  ########

    flair_temp_arr_sq = normalization(flair_temp_arr_sq)
    flair_temp_arr_sq = standardization(flair_temp_arr_sq)

    t1_temp_arr_sq = normalization(t1_temp_arr_sq)
    t1_temp_arr_sq = standardization(t1_temp_arr_sq)

    t1ce_temp_arr_sq = normalization(t1ce_temp_arr_sq)
    t1ce_temp_arr_sq = standardization(t1ce_temp_arr_sq)

    t2_temp_arr_sq = normalization(t2_temp_arr_sq)
    t2_temp_arr_sq = standardization(t2_temp_arr_sq)

    ###################################################

    for i in range(155):
        image_flair[i, :, :, 0] = flair_temp_arr_sq[:, :, i]
        image_t1[i, :, :, 0] = t1_temp_arr_sq[:, :, i]
        image_t1ce[i, :, :, 0] = t1ce_temp_arr_sq[:, :, i]
        image_t2[i, :, :, 0] = t2_temp_arr_sq[:, :, i]

        image_seg[:, :, i] = seg_temp_arr_sq[:, :, i]

    batch_flair = np.zeros((1, img_H, img_W, 1), dtype=np.float)
    batch_t1 = np.zeros((1, img_H, img_W, 1), dtype=np.float)
    batch_t1ce = np.zeros((1, img_H, img_W, 1), dtype=np.float)
    batch_t2 = np.zeros((1, img_H, img_W, 1), dtype=np.float)

    for i in range(155):
        batch_flair[0, :, :, 0] = image_flair[i, :, :, 0]
        batch_t1[0, :, :, 0] = image_t1[i, :, :, 0]
        batch_t1ce[0, :, :, 0] = image_t1ce[i, :, :, 0]
        batch_t2[0, :, :, 0] = image_t2[i, :, :, 0]

        t_predict_image = sess.run(predict_image, feed_dict={flair_in: batch_flair, t1_in: batch_t1,
                                                             t1ce_in: batch_t1ce, t2_in: batch_t2,
                                                             tst: True})
        t_predict_image_p = np.squeeze(t_predict_image)
        t_predict_image_p[t_predict_image_p == 3] = 4
        image_predict[img_H_s:img_H_e, img_W_s:img_W_e, i] = t_predict_image_p[:, :]

    image_predict = image_predict.astype(int16)
    new_image_predict = nib.Nifti1Image(image_predict, affine_array)
    nib.save(new_image_predict, os.path.join(output_dir_path + patient_path[4:len(patient_path)] + '.nii.gz'))

    dice_ET, dice_WT, dice_TC = calculate_dice(image_predict, image_seg, patient_path)
    return dice_ET, dice_WT, dice_TC


output_path_tts = './TTS_' + branch_name + '_submit/'
if not os.path.exists(output_path_tts):
    os.makedirs(output_path_tts)

dice_trainingset = np.zeros((training_set.num_floder, 3), dtype=np.float)  # Dice_ET, Dice_WT, Dice_TC

print('dice:')
print('patient_ID, Dice_ET, Dice_WT, Dice_TC')

for j in range(training_set.num_floder):
    dice_trainingset[j, 0], dice_trainingset[j, 1], dice_trainingset[j, 2] = brats_test_trainingset(
        training_set.dir_path, output_path_tts, training_set.brats_name[j])

np.savetxt(('training_' + branch_name + '_DiceOfTrainingSet.txt'), dice_trainingset, fmt='%f')
print('mean dice:')
print(np.mean(dice_trainingset[:, 0]), np.mean(dice_trainingset[:, 1]), np.mean(dice_trainingset[:, 2]))


######################################
# test training-set, cut
#######################################

def brats_test_trainingset_cut(input_dir_path, output_dir_path, patient_path):
    image_flair = np.zeros((155, img_H, img_W, 1), dtype=np.float)
    image_t1 = np.zeros((155, img_H, img_W, 1), dtype=np.float)
    image_t1ce = np.zeros((155, img_H, img_W, 1), dtype=np.float)
    image_t2 = np.zeros((155, img_H, img_W, 1), dtype=np.float)

    image_predict = np.zeros((240, 240, 155), dtype=np.float)
    image_seg = np.zeros((240, 240, 155), dtype=np.float)

    zero_pixel_percent = np.zeros(155, dtype=np.float)

    path_flair = os.path.join(input_dir_path + patient_path + '/' + patient_path[4:len(patient_path)] + '_flair.nii.gz')
    path_t1 = os.path.join(input_dir_path + patient_path + '/' + patient_path[4:len(patient_path)] + '_t1.nii.gz')
    path_t1ce = os.path.join(input_dir_path + patient_path + '/' + patient_path[4:len(patient_path)] + '_t1ce.nii.gz')
    path_t2 = os.path.join(input_dir_path + patient_path + '/' + patient_path[4:len(patient_path)] + '_t2.nii.gz')
    path_seg = os.path.join(input_dir_path + patient_path + '/' + patient_path[4:len(patient_path)] + '_seg.nii.gz')

    flair_temp = nib.load(path_flair)
    flair_temp_arr = flair_temp.get_fdata()
    flair_temp_arr_sq = np.squeeze(flair_temp_arr)
    flair_temp_arr_sq = extract_patch(flair_temp_arr_sq, img_H, img_W, img_H_s, img_H_e, img_W_s, img_W_e)

    t1_temp = nib.load(path_t1)
    t1_temp_arr = t1_temp.get_fdata()
    t1_temp_arr_sq = np.squeeze(t1_temp_arr)
    t1_temp_arr_sq = extract_patch(t1_temp_arr_sq, img_H, img_W, img_H_s, img_H_e, img_W_s, img_W_e)

    t1ce_temp = nib.load(path_t1ce)
    t1ce_temp_arr = t1ce_temp.get_fdata()
    t1ce_temp_arr_sq = np.squeeze(t1ce_temp_arr)
    t1ce_temp_arr_sq = extract_patch(t1ce_temp_arr_sq, img_H, img_W, img_H_s, img_H_e, img_W_s, img_W_e)

    t2_temp = nib.load(path_t2)
    t2_temp_arr = t2_temp.get_fdata()
    t2_temp_arr_sq = np.squeeze(t2_temp_arr)
    t2_temp_arr_sq = extract_patch(t2_temp_arr_sq, img_H, img_W, img_H_s, img_H_e, img_W_s, img_W_e)

    seg_temp = nib.load(path_seg)
    seg_temp_arr = seg_temp.get_fdata()
    seg_temp_arr_sq = np.squeeze(seg_temp_arr)

    affine_array = flair_temp.affine

    ###################### calculate cutting condition

    background_flair = np.zeros(155, dtype=np.float)
    for k in range(155):
        background_flair[k] = count_pixel(flair_temp_arr_sq[:, :, k], 0)
        background_flair[k] = background_flair[k] / img_H / img_W

    contour_ssim_flair = np.zeros(155, dtype=np.float)
    signal_ssim_flair = np.zeros(155, dtype=np.float)

    for i in range(155):
        contour_ssim_flair[i], signal_ssim_flair[i] = count_contour_ssim(flair_temp_arr_sq[:, :, i])

    ######################

    ######  Normalization and Standardization  ########

    flair_temp_arr_sq = normalization(flair_temp_arr_sq)
    flair_temp_arr_sq = standardization(flair_temp_arr_sq)

    t1_temp_arr_sq = normalization(t1_temp_arr_sq)
    t1_temp_arr_sq = standardization(t1_temp_arr_sq)

    t1ce_temp_arr_sq = normalization(t1ce_temp_arr_sq)
    t1ce_temp_arr_sq = standardization(t1ce_temp_arr_sq)

    t2_temp_arr_sq = normalization(t2_temp_arr_sq)
    t2_temp_arr_sq = standardization(t2_temp_arr_sq)

    ###################################################

    for i in range(155):
        image_flair[i, :, :, 0] = flair_temp_arr_sq[:, :, i]
        image_t1[i, :, :, 0] = t1_temp_arr_sq[:, :, i]
        image_t1ce[i, :, :, 0] = t1ce_temp_arr_sq[:, :, i]
        image_t2[i, :, :, 0] = t2_temp_arr_sq[:, :, i]

        image_seg[:, :, i] = seg_temp_arr_sq[:, :, i]

    batch_flair = np.zeros((1, img_H, img_W, 1), dtype=np.float)
    batch_t1 = np.zeros((1, img_H, img_W, 1), dtype=np.float)
    batch_t1ce = np.zeros((1, img_H, img_W, 1), dtype=np.float)
    batch_t2 = np.zeros((1, img_H, img_W, 1), dtype=np.float)

    number_slide = 0

    for i in range(155):
        if i <= 77 and background_flair[i] <= 0.90 and contour_ssim_flair[i] >= 0.35 and signal_ssim_flair[
            i] <= 0.58:
            batch_flair[0, :, :, 0] = image_flair[i, :, :, 0]
            batch_t1[0, :, :, 0] = image_t1[i, :, :, 0]
            batch_t1ce[0, :, :, 0] = image_t1ce[i, :, :, 0]
            batch_t2[0, :, :, 0] = image_t2[i, :, :, 0]

            t_predict_image = sess.run(predict_image, feed_dict={flair_in: batch_flair, t1_in: batch_t1,
                                                                 t1ce_in: batch_t1ce, t2_in: batch_t2,
                                                                 tst: True})
            t_predict_image_p = np.squeeze(t_predict_image)
            t_predict_image_p[t_predict_image_p == 3] = 4
            image_predict[img_H_s:img_H_e, img_W_s:img_W_e, i] = t_predict_image_p[:, :]

            number_slide = number_slide + 1

        if i > 77 and background_flair[i] < 1 and signal_ssim_flair[i] <= 0.58:
            batch_flair[0, :, :, 0] = image_flair[i, :, :, 0]
            batch_t1[0, :, :, 0] = image_t1[i, :, :, 0]
            batch_t1ce[0, :, :, 0] = image_t1ce[i, :, :, 0]
            batch_t2[0, :, :, 0] = image_t2[i, :, :, 0]

            t_predict_image = sess.run(predict_image, feed_dict={flair_in: batch_flair, t1_in: batch_t1,
                                                                 t1ce_in: batch_t1ce, t2_in: batch_t2,
                                                                 tst: True})
            t_predict_image_p = np.squeeze(t_predict_image)
            t_predict_image_p[t_predict_image_p == 3] = 4
            image_predict[img_H_s:img_H_e, img_W_s:img_W_e, i] = t_predict_image_p[:, :]

            number_slide = number_slide + 1

    image_predict = image_predict.astype(int16)
    new_image_predict = nib.Nifti1Image(image_predict, affine_array)
    nib.save(new_image_predict, os.path.join(output_dir_path + patient_path[4:len(patient_path)] + '.nii.gz'))

    dice_ET, dice_WT, dice_TC = calculate_dice(image_predict, image_seg, patient_path)
    return dice_ET, dice_WT, dice_TC, number_slide


output_path_tts_cut = './TTS_' + branch_name + '_submit_Cut/'
if not os.path.exists(output_path_tts_cut):
    os.makedirs(output_path_tts_cut)

dice_trainingset_cut = np.zeros((training_set.num_floder, 3), dtype=np.float)  # Dice_ET, Dice_WT, Dice_TC

print('dice (cut):')
print('patient_ID, Dice_ET, Dice_WT, Dice_TC')

slide_number_tts = np.zeros(training_set.num_floder, dtype=np.int)

for j in range(training_set.num_floder):
    dice_trainingset_cut[j, 0], dice_trainingset_cut[j, 1], dice_trainingset_cut[j, 2], slide_number_tts[
        j] = brats_test_trainingset_cut(
        training_set.dir_path, output_path_tts_cut, training_set.brats_name[j])

np.savetxt(('training_' + branch_name + '_DiceOfTrainingSet_cut.txt'), dice_trainingset_cut, fmt='%f')
print('mean dice (cut):')
print(np.mean(dice_trainingset_cut[:, 0]), np.mean(dice_trainingset_cut[:, 1]), np.mean(dice_trainingset_cut[:, 2]))

for j in range(training_set.num_floder):
    print(training_set.brats_name[j], slide_number_tts[j])
print('Average number of slides: ', np.mean(slide_number_tts))


######################################
# test training-set, cut, post-processing
#######################################

def brats_test_trainingset_cut_post(input_dir_path, output_dir_path, patient_path):
    image_flair = np.zeros((155, img_H, img_W, 1), dtype=np.float)
    image_t1 = np.zeros((155, img_H, img_W, 1), dtype=np.float)
    image_t1ce = np.zeros((155, img_H, img_W, 1), dtype=np.float)
    image_t2 = np.zeros((155, img_H, img_W, 1), dtype=np.float)

    image_predict = np.zeros((240, 240, 155), dtype=np.float)
    image_seg = np.zeros((240, 240, 155), dtype=np.float)

    zero_pixel_percent = np.zeros(155, dtype=np.float)

    path_flair = os.path.join(input_dir_path + patient_path + '/' + patient_path[4:len(patient_path)] + '_flair.nii.gz')
    path_t1 = os.path.join(input_dir_path + patient_path + '/' + patient_path[4:len(patient_path)] + '_t1.nii.gz')
    path_t1ce = os.path.join(input_dir_path + patient_path + '/' + patient_path[4:len(patient_path)] + '_t1ce.nii.gz')
    path_t2 = os.path.join(input_dir_path + patient_path + '/' + patient_path[4:len(patient_path)] + '_t2.nii.gz')
    path_seg = os.path.join(input_dir_path + patient_path + '/' + patient_path[4:len(patient_path)] + '_seg.nii.gz')

    flair_temp = nib.load(path_flair)
    flair_temp_arr = flair_temp.get_fdata()
    flair_temp_arr_sq = np.squeeze(flair_temp_arr)
    flair_temp_arr_sq = extract_patch(flair_temp_arr_sq, img_H, img_W, img_H_s, img_H_e, img_W_s, img_W_e)

    t1_temp = nib.load(path_t1)
    t1_temp_arr = t1_temp.get_fdata()
    t1_temp_arr_sq = np.squeeze(t1_temp_arr)
    t1_temp_arr_sq = extract_patch(t1_temp_arr_sq, img_H, img_W, img_H_s, img_H_e, img_W_s, img_W_e)

    t1ce_temp = nib.load(path_t1ce)
    t1ce_temp_arr = t1ce_temp.get_fdata()
    t1ce_temp_arr_sq = np.squeeze(t1ce_temp_arr)
    t1ce_temp_arr_sq = extract_patch(t1ce_temp_arr_sq, img_H, img_W, img_H_s, img_H_e, img_W_s, img_W_e)

    t2_temp = nib.load(path_t2)
    t2_temp_arr = t2_temp.get_fdata()
    t2_temp_arr_sq = np.squeeze(t2_temp_arr)
    t2_temp_arr_sq = extract_patch(t2_temp_arr_sq, img_H, img_W, img_H_s, img_H_e, img_W_s, img_W_e)

    seg_temp = nib.load(path_seg)
    seg_temp_arr = seg_temp.get_fdata()
    seg_temp_arr_sq = np.squeeze(seg_temp_arr)

    affine_array = flair_temp.affine

    ###################### calculate cutting condition

    background_flair = np.zeros(155, dtype=np.float)
    for k in range(155):
        background_flair[k] = count_pixel(flair_temp_arr_sq[:, :, k], 0)
        background_flair[k] = background_flair[k] / img_H / img_W

    contour_ssim_flair = np.zeros(155, dtype=np.float)
    signal_ssim_flair = np.zeros(155, dtype=np.float)

    for i in range(155):
        contour_ssim_flair[i], signal_ssim_flair[i] = count_contour_ssim(flair_temp_arr_sq[:, :, i])

    ######################

    ######  Normalization and Standardization  ########

    flair_temp_arr_sq = normalization(flair_temp_arr_sq)
    flair_temp_arr_sq = standardization(flair_temp_arr_sq)

    t1_temp_arr_sq = normalization(t1_temp_arr_sq)
    t1_temp_arr_sq = standardization(t1_temp_arr_sq)

    t1ce_temp_arr_sq = normalization(t1ce_temp_arr_sq)
    t1ce_temp_arr_sq = standardization(t1ce_temp_arr_sq)

    t2_temp_arr_sq = normalization(t2_temp_arr_sq)
    t2_temp_arr_sq = standardization(t2_temp_arr_sq)

    ###################################################

    for i in range(155):
        image_flair[i, :, :, 0] = flair_temp_arr_sq[:, :, i]
        image_t1[i, :, :, 0] = t1_temp_arr_sq[:, :, i]
        image_t1ce[i, :, :, 0] = t1ce_temp_arr_sq[:, :, i]
        image_t2[i, :, :, 0] = t2_temp_arr_sq[:, :, i]

        image_seg[:, :, i] = seg_temp_arr_sq[:, :, i]

    batch_flair = np.zeros((1, img_H, img_W, 1), dtype=np.float)
    batch_t1 = np.zeros((1, img_H, img_W, 1), dtype=np.float)
    batch_t1ce = np.zeros((1, img_H, img_W, 1), dtype=np.float)
    batch_t2 = np.zeros((1, img_H, img_W, 1), dtype=np.float)

    number_slide = 0

    for i in range(155):
        if i <= 77 and background_flair[i] <= 0.90 and contour_ssim_flair[i] >= 0.35 and signal_ssim_flair[
            i] <= 0.58:
            batch_flair[0, :, :, 0] = image_flair[i, :, :, 0]
            batch_t1[0, :, :, 0] = image_t1[i, :, :, 0]
            batch_t1ce[0, :, :, 0] = image_t1ce[i, :, :, 0]
            batch_t2[0, :, :, 0] = image_t2[i, :, :, 0]

            t_predict_image = sess.run(predict_image, feed_dict={flair_in: batch_flair, t1_in: batch_t1,
                                                                 t1ce_in: batch_t1ce, t2_in: batch_t2,
                                                                 tst: True})
            t_predict_image_p = np.squeeze(t_predict_image)
            t_predict_image_p[t_predict_image_p == 3] = 4
            image_predict[img_H_s:img_H_e, img_W_s:img_W_e, i] = t_predict_image_p[:, :]

            number_slide = number_slide + 1

        if i > 77 and background_flair[i] < 1 and signal_ssim_flair[i] <= 0.58:
            batch_flair[0, :, :, 0] = image_flair[i, :, :, 0]
            batch_t1[0, :, :, 0] = image_t1[i, :, :, 0]
            batch_t1ce[0, :, :, 0] = image_t1ce[i, :, :, 0]
            batch_t2[0, :, :, 0] = image_t2[i, :, :, 0]

            t_predict_image = sess.run(predict_image, feed_dict={flair_in: batch_flair, t1_in: batch_t1,
                                                                 t1ce_in: batch_t1ce, t2_in: batch_t2,
                                                                 tst: True})
            t_predict_image_p = np.squeeze(t_predict_image)
            t_predict_image_p[t_predict_image_p == 3] = 4
            image_predict[img_H_s:img_H_e, img_W_s:img_W_e, i] = t_predict_image_p[:, :]

            number_slide = number_slide + 1

    image_post = post_processing(image_predict, 7, 6, 1)

    image_post = image_post.astype(int16)
    new_image_post = nib.Nifti1Image(image_post, affine_array)
    nib.save(new_image_post, os.path.join(output_dir_path + patient_path[4:len(patient_path)] + '.nii.gz'))

    dice_ET, dice_WT, dice_TC = calculate_dice(image_post, image_seg, patient_path)
    return dice_ET, dice_WT, dice_TC, number_slide


output_path_tts_cut_post = './TTS_' + branch_name + '_submit_CutPost/'
if not os.path.exists(output_path_tts_cut_post):
    os.makedirs(output_path_tts_cut_post)

dice_trainingset_cut_post = np.zeros((training_set.num_floder, 3), dtype=np.float)  # Dice_ET, Dice_WT, Dice_TC

print('dice (cut and post-processing):')
print('patient_ID, Dice_ET, Dice_WT, Dice_TC')

slide_number_tts_post = np.zeros(training_set.num_floder, dtype=np.int)

for j in range(training_set.num_floder):
    dice_trainingset_cut_post[j, 0], dice_trainingset_cut_post[j, 1], dice_trainingset_cut_post[j, 2], \
    slide_number_tts_post[
        j] = brats_test_trainingset_cut_post(
        training_set.dir_path, output_path_tts_cut_post, training_set.brats_name[j])

np.savetxt(('training_' + branch_name + '_DiceOfTrainingSet_cutpost.txt'), dice_trainingset_cut_post, fmt='%f')
print('mean dice (cut and post-processing):')
print(np.mean(dice_trainingset_cut_post[:, 0]), np.mean(dice_trainingset_cut_post[:, 1]),
      np.mean(dice_trainingset_cut_post[:, 2]))

for j in range(training_set.num_floder):
    print(training_set.brats_name[j], slide_number_tts_post[j])
print('Average number of slides: ', np.mean(slide_number_tts_post))




2021-05-30 00:36:37
Start loading dataset: 
rate of progress 25%
rate of progress 50%
rate of progress 75%
rate of progress 100%
Start training: 
Instructions for updating:
Colocations handled automatically by placer.
Instructions for updating:

Future major versions of TensorFlow will allow gradients to flow
into the labels input on backprop by default.

See `tf.nn.softmax_cross_entropy_with_logits_v2`.

Instructions for updating:
Use tf.cast instead.
step 0, training accuracy 0.399873, cross entropy 182452, learning rate 0.01
step 50, training accuracy 0.985614, cross entropy 4182.39, learning rate 0.00999984
step 100, training accuracy 0.989076, cross entropy 3205.2, learning rate 0.00999938
step 150, training accuracy 0.99024, cross entropy 2803.74, learning rate 0.0099986
step 200, training accuracy 0.986281, cross entropy 3808.17, learning rate 0.00999751
step 250, training accuracy 0.990579, cross entropy 2749.17, learning rate 0.00999611
step 300, training accuracy 0.990218, cr

step 4150, training accuracy 0.992828, cross entropy 2049.72, learning rate 0.00896487
step 4200, training accuracy 0.993804, cross entropy 1706.08, learning rate 0.0089407
step 4250, training accuracy 0.994209, cross entropy 1621.61, learning rate 0.00891629
step 4300, training accuracy 0.992848, cross entropy 1902.02, learning rate 0.00889163
step 4350, training accuracy 0.993034, cross entropy 1904.4, learning rate 0.00886673
step 4400, training accuracy 0.994824, cross entropy 1434.97, learning rate 0.00884159
step 4450, training accuracy 0.993965, cross entropy 1709.85, learning rate 0.00881621
step 4500, training accuracy 0.991364, cross entropy 2351.2, learning rate 0.00879059
step 4550, training accuracy 0.993195, cross entropy 1844.41, learning rate 0.00876474
step 4600, training accuracy 0.993765, cross entropy 1645.97, learning rate 0.00873865
step 4650, training accuracy 0.994215, cross entropy 1561.93, learning rate 0.00871233
step 4700, training accuracy 0.994028, cross e

step 8900, training accuracy 0.994146, cross entropy 1686.81, learning rate 0.00582544
step 8950, training accuracy 0.993731, cross entropy 1715.53, learning rate 0.00578649
step 9000, training accuracy 0.993642, cross entropy 1812.38, learning rate 0.0057475
step 9050, training accuracy 0.994557, cross entropy 1541.35, learning rate 0.00570845
step 9100, training accuracy 0.994478, cross entropy 1464.55, learning rate 0.00566936
step 9150, training accuracy 0.993475, cross entropy 1758.77, learning rate 0.00563023
step 9200, training accuracy 0.991748, cross entropy 2277.92, learning rate 0.00559107
step 9250, training accuracy 0.994085, cross entropy 1644.68, learning rate 0.00555186
step 9300, training accuracy 0.993611, cross entropy 1687.74, learning rate 0.00551262
step 9350, training accuracy 0.993341, cross entropy 1783.59, learning rate 0.00547335
step 9400, training accuracy 0.995125, cross entropy 1314.97, learning rate 0.00543405
step 9450, training accuracy 0.99506, cross 

step 13600, training accuracy 0.993229, cross entropy 1784.4, learning rate 0.00227647
step 13650, training accuracy 0.994491, cross entropy 1532.83, learning rate 0.00224347
step 13700, training accuracy 0.994496, cross entropy 1500.06, learning rate 0.00221063
step 13750, training accuracy 0.994089, cross entropy 1566.86, learning rate 0.00217797
step 13800, training accuracy 0.994338, cross entropy 1508.05, learning rate 0.00214549
step 13850, training accuracy 0.995673, cross entropy 1209.84, learning rate 0.00211318
step 13900, training accuracy 0.99413, cross entropy 1551.33, learning rate 0.00208106
step 13950, training accuracy 0.994919, cross entropy 1343.75, learning rate 0.00204911
step 14000, training accuracy 0.994364, cross entropy 1487.66, learning rate 0.00201735
step 14050, training accuracy 0.99368, cross entropy 1650.28, learning rate 0.00198578
step 14100, training accuracy 0.993582, cross entropy 1711.45, learning rate 0.00195439
step 14150, training accuracy 0.995

step 18250, training accuracy 0.995575, cross entropy 1176.55, learning rate 0.000169656
step 18300, training accuracy 0.995353, cross entropy 1282.92, learning rate 0.000159642
step 18350, training accuracy 0.993419, cross entropy 1736.41, learning rate 0.000149931
step 18400, training accuracy 0.995056, cross entropy 1406.91, learning rate 0.000140522
step 18450, training accuracy 0.994261, cross entropy 1531.12, learning rate 0.000131415
step 18500, training accuracy 0.995101, cross entropy 1271.68, learning rate 0.000122612
step 18550, training accuracy 0.994239, cross entropy 1595.17, learning rate 0.000114113
step 18600, training accuracy 0.994187, cross entropy 1529.03, learning rate 0.000105918
step 18650, training accuracy 0.994754, cross entropy 1376.61, learning rate 9.80287e-05
step 18700, training accuracy 0.992356, cross entropy 2093.11, learning rate 9.04442e-05
step 18750, training accuracy 0.995301, cross entropy 1254.9, learning rate 8.3166e-05
step 18800, training ac

...

LGG/Brats18_2013_6_1 0.6343683751051604 0.9567309259196957 0.8411703189387282
HGG/Brats18_CBICA_AXJ_1 0.7652334255269246 0.9150577126483938 0.7245337529323277
HGG/Brats18_TCIA04_192_1 0.9791715111418159 0.9641163571940309 0.9680353430321984
LGG/Brats18_TCIA10_420_1 0.2818532818351446 0.839135333885602 0.6880182290003839
LGG/Brats18_TCIA10_637_1 0.4736268865649039 0.9189863937084494 0.7695898334985485
HGG/Brats18_CBICA_AQV_1 0.8816692877109368 0.9110714671848208 0.9290644272380815
LGG/Brats18_2013_24_1 1 0.9280886872636479 0.9398877785964055
HGG/Brats18_CBICA_ASO_1 0.8732808649301483 0.929069721818602 0.9288957019016305
LGG/Brats18_TCIA12_298_1 1 0.953822253641176 0.9219404117799826
LGG/Brats18_TCIA12_249_1 0.08641975305974699 0.9640091576319417 0.9334288466756141
LGG/Brats18_TCIA10_299_1 0.6620430233886091 0.9407072350086447 0.6580449726164624
LGG/Brats18_2013_28_1 0.8652310180885124 0.8873089842136408 0.779436020746031
HGG/Brats18_CBICA_ALU_1 0.8306065113135794 0.8563692666858147 0.88

LGG/Brats18_TCIA13_650_1 0.0 0.8043040565045901 0.7074809160269339
HGG/Brats18_CBICA_ABB_1 0.8071694030163844 0.7826450819866029 0.8876925377565138
HGG/Brats18_TCIA04_361_1 0.9143626570903893 0.9682591731538672 0.9296459021594069
LGG/Brats18_TCIA10_640_1 0.8574317491549615 0.8740685411426949 0.5965734172092318
HGG/Brats18_CBICA_AQG_1 0.817627449379481 0.8440360315530864 0.8287056223178014
HGG/Brats18_TCIA08_436_1 0.8829462484587126 0.9332831444643263 0.9168310400271578
LGG/Brats18_2013_16_1 1 0.9074631379955331 0.4895658431219769
HGG/Brats18_CBICA_ASH_1 0.7250070801267344 0.6736375719967354 0.862640800987756
LGG/Brats18_TCIA13_633_1 0.38959832829371727 0.9513421951146598 0.8538011059935582
HGG/Brats18_2013_25_1 0.4072883172125093 0.8741617744568442 0.6175006223511482
HGG/Brats18_CBICA_BFB_1 0.8384549356202613 0.9300827966873098 0.9235689045920078
HGG/Brats18_CBICA_AZD_1 0.7919666540256162 0.7801870311335927 0.881489250123392
HGG/Brats18_TCIA02_321_1 0.944560300620642 0.9330792570114211

LGG/Brats18_TCIA10_103_1 0.8954302670516862 0.9117426990489236 0.7768814968750363
LGG/Brats18_TCIA12_470_1 0.8571021512913933 0.9155108469214156 0.7384558890963145
HGG/Brats18_TCIA02_171_1 0.8830452762073687 0.9307465825437111 0.8595190380675398
LGG/Brats18_TCIA13_624_1 0.8760345778881918 0.9685091039242758 0.8418422299735391
HGG/Brats18_CBICA_ASU_1 0.843615367529009 0.8961948008160693 0.9023268932424134
HGG/Brats18_2013_23_1 0.8266905714148035 0.9520939737858682 0.9579294300112455
HGG/Brats18_TCIA02_368_1 0.8777460439971011 0.9605305002293965 0.939560852255903
HGG/Brats18_TCIA08_319_1 0.7843752536024444 0.9587634838414512 0.9399962356466403
HGG/Brats18_TCIA02_331_1 0.948997827937233 0.931587794059244 0.9594319951409205
HGG/Brats18_2013_20_1 0.8711488279270321 0.9378282711613857 0.8973697550992579
LGG/Brats18_2013_0_1 0.8695853736248093 0.8829754707109421 0.8095870648886466
LGG/Brats18_TCIA10_387_1 0.7893462469351407 0.9194590784094584 0.854178097665773
LGG/Brats18_TCIA13_653_1 0.84980

LGG/Brats18_TCIA10_307_1 1 0.9437191110027326 0.793709503597885
HGG/Brats18_TCIA02_377_1 0.9555389280500969 0.9670576605723743 0.9633044343740771
HGG/Brats18_TCIA05_396_1 0.8764486134189282 0.9594144917687408 0.8570523923919148
HGG/Brats18_CBICA_ANI_1 0.8376026473781247 0.9263481245045831 0.8598137312576262
LGG/Brats18_TCIA10_410_1 0.8612250573583846 0.920093675230862 0.7933002737741304
HGG/Brats18_CBICA_AXL_1 0.8916691277137868 0.8382816136115564 0.9297844827576187
HGG/Brats18_TCIA01_203_1 0.8820746965886042 0.9458776672342103 0.9178359329108491
HGG/Brats18_TCIA03_375_1 0.8962105471437692 0.9145679563554893 0.9261175794150839
LGG/Brats18_TCIA10_202_1 0.0 0.9287350836993727 0.9004535298113273
HGG/Brats18_CBICA_AQJ_1 0.8729385748209166 0.9021232161498421 0.9097314591780031
LGG/Brats18_TCIA13_630_1 1 0.9516734988910379 0.8401987060194946
HGG/Brats18_CBICA_AQD_1 0.879394166035159 0.9324177752249558 0.9308911588521952
LGG/Brats18_TCIA13_615_1 0.2964980544708625 0.9310755037626726 0.8500064

HGG/Brats18_CBICA_APR_1 0.9002714298196297 0.9178129804151341 0.9662002270631181
HGG/Brats18_CBICA_ARZ_1 0.7484326018515505 0.7487714987691991 0.7102866778849819
HGG/Brats18_TCIA01_401_1 0.9385159449053454 0.9548317991629668 0.9644388892965677
HGG/Brats18_CBICA_AYU_1 0.8429055325491689 0.8752466564339824 0.862038664319166
HGG/Brats18_TCIA02_151_1 0.9224445196011072 0.9687250144098433 0.9708536685102778
HGG/Brats18_CBICA_AQT_1 0.8355475040229617 0.9110182737940401 0.9223865974313268
LGG/Brats18_2013_15_1 1 0.880713152525677 0.6658427607123696
HGG/Brats18_CBICA_AUQ_1 0.693501454881341 0.8601331360914936 0.736316568033722
HGG/Brats18_TCIA01_150_1 0.91239369859067 0.945716956439749 0.9273741543237536
HGG/Brats18_CBICA_AWG_1 0.8425100506874934 0.9343742838848098 0.8933329153984055
HGG/Brats18_TCIA01_429_1 0.939152031220893 0.939711035845974 0.9608436317305463
HGG/Brats18_CBICA_AUN_1 0.9107888916181636 0.9445618278288263 0.965855455081978
LGG/Brats18_TCIA09_462_1 1 0.9272402697894059 0.73746